# Image Compression with K-Means and Dimensionality Reduction with PCA

## Part 1: Image Compression with K-Means

### 1.1 Load and Preprocess the Image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import urllib.request

url_bird = "https://github.com/fengdu78/Coursera-ML-AndrewNg-Notes/raw/master/code/ex7-kmeans%20and%20PCA/data/bird_small.mat"
urllib.request.urlretrieve(url_bird, "bird_small.mat")

mat = scipy.io.loadmat("bird_small.mat")
A = mat["A"]

print(f"Image shape: {A.shape}")

# Normalize pixel values to [0, 1]
A = A / 255.0

# Reshape into (m, 3) where each row is one pixel's RGB values
img_size = A.shape
X = A.reshape(-1, 3)

print(f"Reshaped data: {X.shape} ({X.shape[0]} pixels, 3 RGB channels)")

plt.figure(figsize=(5, 5))
plt.imshow(A)
plt.title("Original Image")
plt.axis("off")
plt.tight_layout()
plt.show()

### 1.2 K-Means Helper Functions

In [ ]:
def find_closest_centroids(X, centroids):
    m = X.shape[0]
    idx = np.zeros(m, dtype=int)
    for i in range(m):
        distances = np.linalg.norm(X[i] - centroids, axis=1)
        idx[i] = np.argmin(distances)
    return idx


def compute_centroids(X, idx, K):
    n = X.shape[1]
    centroids = np.zeros((K, n))
    for k in range(K):
        points = X[idx == k]
        if len(points) > 0:
            centroids[k] = points.mean(axis=0)
    return centroids


def init_centroids(X, K):
    indices = np.random.choice(X.shape[0], size=K, replace=False)
    return X[indices].copy()


def run_k_means(X, initial_centroids, max_iters=10):
    K = initial_centroids.shape[0]
    centroids = initial_centroids.copy()
    idx = None
    for _ in range(max_iters):
        idx = find_closest_centroids(X, centroids)
        centroids = compute_centroids(X, idx, K)
    return idx, centroids


print("K-Means functions defined.")

### 1.3 Apply K-Means to Compress the Image

In [ ]:
K = 16
max_iters = 10

np.random.seed(42)
initial_centroids = init_centroids(X, K)

print(f"Running K-Means with K={K} colors and {max_iters} iterations...")
idx, centroids = run_k_means(X, initial_centroids, max_iters)
print("Done.")
print(f"Final centroids shape: {centroids.shape}")

### 1.4 Recover and Display the Compressed Image

In [ ]:
# Replace each pixel with the color of its assigned centroid
X_compressed = centroids[idx]

# Reshape back to original image dimensions
img_compressed = X_compressed.reshape(img_size)
img_compressed = np.clip(img_compressed, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(A)
axes[0].set_title(f"Original Image\n(unique colors: {len(np.unique(X.reshape(-1,3), axis=0))}+)")
axes[0].axis("off")

axes[1].imshow(img_compressed)
axes[1].set_title(f"Compressed Image\n({K} colors, {max_iters} iterations)")
axes[1].axis("off")

plt.suptitle("Image Compression via K-Means Color Quantization", fontsize=13)
plt.tight_layout()
plt.show()

original_bits = X.shape[0] * 24
compressed_bits = X.shape[0] * int(np.ceil(np.log2(K))) + K * 24
print(f"Original size:   {original_bits:,} bits")
print(f"Compressed size: {compressed_bits:,} bits")
print(f"Compression ratio: {original_bits / compressed_bits:.2f}x")

## Part 2: Dimensionality Reduction with PCA

### 2.1 Load the Dataset

In [ ]:
url_pca = "https://github.com/fengdu78/Coursera-ML-AndrewNg-Notes/raw/master/code/ex7-kmeans%20and%20PCA/data/ex7data1.mat"
urllib.request.urlretrieve(url_pca, "ex7data1.mat")

mat2 = scipy.io.loadmat("ex7data1.mat")
X2 = mat2["X"]

print(f"PCA dataset shape: {X2.shape}")

plt.figure(figsize=(6, 5))
plt.scatter(X2[:, 0], X2[:, 1], s=20, alpha=0.7, color="steelblue")
plt.title("ex7data1 — Raw Data")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.axis("equal")
plt.tight_layout()
plt.show()

### 2.2 Feature Normalization

In [ ]:
def feature_normalize(X):
    """
    Normalizes each feature to have zero mean and unit variance.

    Returns:
        X_norm : normalized data
        mu     : mean of each feature
        sigma  : standard deviation of each feature
    """
    mu = X.mean(axis=0)
    sigma = X.std(axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma


X2_norm, mu, sigma = feature_normalize(X2)

print(f"Mean before normalization:  {X2.mean(axis=0)}")
print(f"Mean after normalization:   {X2_norm.mean(axis=0).round(6)}")
print(f"Std after normalization:    {X2_norm.std(axis=0).round(6)}")

### 2.3 Implement PCA

In [ ]:
def run_pca(X_norm):
    """
    Computes PCA on the normalized dataset using Singular Value Decomposition.

    Parameters:
        X_norm : (m, n) normalized data matrix

    Returns:
        U : (n, n) matrix of principal components (columns are eigenvectors)
        S : (n,) singular values (related to explained variance)
    """
    m = X_norm.shape[0]
    covariance_matrix = (1 / m) * X_norm.T @ X_norm
    U, S, Vt = np.linalg.svd(covariance_matrix)
    return U, S


U, S = run_pca(X2_norm)

print("Top principal components (columns of U):")
print(U)
print(f"\nSingular values: {S}")
print(f"Variance explained by PC1: {S[0]/S.sum()*100:.1f}%")

### 2.4 Project Data onto the First Principal Component

In [ ]:
def project_data(X_norm, U, K):
    """
    Projects the data onto the first K principal components.

    Parameters:
        X_norm : (m, n) normalized data
        U      : (n, n) principal components matrix
        K      : number of components to keep

    Returns:
        Z : (m, K) reduced-dimensionality data
    """
    U_reduced = U[:, :K]
    Z = X_norm @ U_reduced
    return Z


K = 1
Z = project_data(X2_norm, U, K)

print(f"Projected data shape: {Z.shape}")
print(f"First 5 projected values: {Z[:5].flatten()}")

### 2.5 Recover the Data from Reduced Dimensionality

In [ ]:
def recover_data(Z, U, K):
    """
    Approximates the original data by projecting back from K dimensions.

    Parameters:
        Z : (m, K) reduced data
        U : (n, n) principal components matrix
        K : number of components used

    Returns:
        X_rec : (m, n) approximate reconstruction of the original data
    """
    U_reduced = U[:, :K]
    X_rec = Z @ U_reduced.T
    return X_rec


X_rec = recover_data(Z, U, K)

print(f"Recovered data shape: {X_rec.shape}")

### 2.6 Visualize Original vs Recovered Data

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(X2_norm[:, 0], X2_norm[:, 1],
            s=25, color="steelblue", alpha=0.7, label="Original (normalized)")

plt.scatter(X_rec[:, 0], X_rec[:, 1],
            s=25, color="tomato", alpha=0.7, label="Recovered (from PC1)")

# Draw lines connecting each point to its projection
for i in range(len(X2_norm)):
    plt.plot([X2_norm[i, 0], X_rec[i, 0]],
             [X2_norm[i, 1], X_rec[i, 1]],
             color="gray", linewidth=0.5, alpha=0.5)

# Plot the first principal component direction
scale = 3
plt.annotate("", xy=(U[0, 0] * scale, U[1, 0] * scale),
             xytext=(0, 0),
             arrowprops=dict(arrowstyle="->", color="black", lw=2))
plt.text(U[0, 0] * scale + 0.1, U[1, 0] * scale, "PC1", fontsize=11)

plt.title("PCA: Original Data vs Projection onto First Principal Component")
plt.xlabel("Feature 1 (normalized)")
plt.ylabel("Feature 2 (normalized)")
plt.axis("equal")
plt.legend()
plt.tight_layout()
plt.show()

reconstruction_error = np.mean(np.sum((X2_norm - X_rec) ** 2, axis=1))
print(f"Mean reconstruction error (1 component): {reconstruction_error:.4f}")